# GRU Incremental Training (New File)

這份 notebook 不會修改原本的 `train_lstm_sequence.ipynb` / `train_gru_sequence.ipynb`。

目標：
- 支援新增類別（例如 155 -> 156）時，不必從零訓練
- 舊模型 warm-start
- 以 replay 資料降低遺忘
- 繼續輸出 Keras / SavedModel / TFLite

In [1]:
import csv
import glob
import os
import sys
from pathlib import Path

import numpy as np
from sklearn.model_selection import train_test_split

CUDA_BIN = Path(r"C:/Program Files/NVIDIA GPU Computing Toolkit/CUDA/v11.5/bin")
CUDNN_BIN_CANDIDATES = [
    Path.cwd() / "third_party" / "cudnn-8.9.7-cuda11" / "bin",
    Path.cwd() / "cv_hands" / "third_party" / "cudnn-8.9.7-cuda11" / "bin",
]

dll_paths = []
if CUDA_BIN.exists():
    dll_paths.append(str(CUDA_BIN))

for candidate in CUDNN_BIN_CANDIDATES:
    if candidate.exists():
        dll_paths.append(str(candidate))
        break

if dll_paths:
    os.environ["PATH"] = ";".join(dll_paths + [os.environ.get("PATH", "")])

print("Python:", sys.executable)
print("CWD:", Path.cwd())
print("DLL search paths:", dll_paths)

import tensorflow as tf

gpus = tf.config.list_physical_devices("GPU")
print("Detected GPUs:", gpus)
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)

Python: c:\Users\User\AppData\Local\Programs\Python\Python310\python.exe
CWD: c:\Users\User\Documents\GitHub\FYP-SignLanguage\cv_hands
DLL search paths: []
Detected GPUs: []


# Config

In [2]:
MODEL_SAVE_PATH = 'model/keypoint_classifier/keypoint_sequence_classifier.keras'
SAVED_MODEL_DIR = 'model/keypoint_classifier/keypoint_sequence_classifier_savedmodel'
TFLITE_SAVE_PATH = 'model/keypoint_classifier/keypoint_sequence_classifier.tflite'
LABEL_PATH = 'Word-Label/keypoint_sequence_classifier_label.csv'
CSV_GLOB = 'Words-Dataset/*_sequence.csv'

SEQUENCE_LENGTH = 25
FEATURES_PER_FRAME = 80

REPLAY_PER_OLD_CLASS = 20
WARMUP_EPOCHS = 5
FINETUNE_EPOCHS = 120
BATCH_SIZE = 32

# Helpers

In [3]:
def load_labels(label_path):
    with open(label_path, encoding='utf-8-sig') as f:
        rows = csv.reader(f)
        labels = [row[0] for row in rows if row]
    return labels


def load_sequence_dataset(csv_glob, sequence_length, features_per_frame):
    csv_files = sorted(glob.glob(csv_glob))
    if not csv_files:
        raise FileNotFoundError(f'No dataset files found by pattern: {csv_glob}')

    x_parts, y_parts = [], []
    expected_feature_cols = sequence_length * features_per_frame

    for csv_file in csv_files:
        data = np.loadtxt(csv_file, delimiter=',', dtype='float32')
        data = np.atleast_2d(data)

        if data.shape[1] < 1 + expected_feature_cols:
            print(f'Skip malformed file: {csv_file}, shape={data.shape}')
            continue

        y = data[:, 0].astype('int32')
        x = data[:, 1:1 + expected_feature_cols].reshape(-1, sequence_length, features_per_frame).astype('float32')

        x_parts.append(x)
        y_parts.append(y)

    if not x_parts:
        raise ValueError('No valid samples loaded from dataset files.')

    X = np.concatenate(x_parts, axis=0)
    y = np.concatenate(y_parts, axis=0)
    return X, y, csv_files


def sanitize_dataset(X, y, num_classes):
    finite_mask = np.isfinite(X).all(axis=(1, 2))
    removed_non_finite = int((~finite_mask).sum())
    if removed_non_finite > 0:
        print(f'Removed non-finite samples: {removed_non_finite}')

    X = X[finite_mask]
    y = y[finite_mask]

    X = np.nan_to_num(X, nan=0.0, posinf=1e3, neginf=-1e3).astype('float32')

    valid_label_mask = (y >= 0) & (y < num_classes)
    removed_invalid_labels = int((~valid_label_mask).sum())
    if removed_invalid_labels > 0:
        print(f'Removed invalid-label samples: {removed_invalid_labels}')

    X = X[valid_label_mask]
    y = y[valid_label_mask]

    return X, y


def build_gru_model(num_classes, sequence_length, features_per_frame):
    from tensorflow.keras import layers

    inputs = tf.keras.Input(shape=(sequence_length, features_per_frame), name='sequence_input')
    x = layers.Bidirectional(layers.GRU(128, return_sequences=True), name='bigru_1')(inputs)
    x = layers.Dropout(0.3, name='dropout_1')(x)
    x = layers.Bidirectional(layers.GRU(64), name='bigru_2')(x)
    x = layers.Dropout(0.3, name='dropout_2')(x)
    x = layers.Dense(64, activation='relu', name='dense_1')(x)
    x = layers.BatchNormalization(name='bn_1')(x)
    x = layers.Dropout(0.3, name='dropout_3')(x)
    outputs = layers.Dense(num_classes, activation='softmax', name='classifier')(x)

    model = tf.keras.Model(inputs=inputs, outputs=outputs, name='gru_sequence_classifier')
    return model


def copy_backbone_weights(old_model, new_model):
    copied = []
    for layer in new_model.layers:
        if layer.name == 'classifier':
            continue
        try:
            old_layer = old_model.get_layer(layer.name)
            layer.set_weights(old_layer.get_weights())
            copied.append(layer.name)
        except Exception:
            pass
    print('Copied backbone layers:', copied)


def expand_classifier_weights(old_model, new_model):
    old_w, old_b = old_model.get_layer('classifier').get_weights()
    new_w, new_b = new_model.get_layer('classifier').get_weights()

    old_classes = old_w.shape[1]
    new_classes = new_w.shape[1]

    if new_classes < old_classes:
        raise ValueError(f'New classes ({new_classes}) < old classes ({old_classes}), cannot expand head.')

    new_w[:, :old_classes] = old_w
    new_b[:old_classes] = old_b

    new_model.get_layer('classifier').set_weights([new_w, new_b])
    print(f'Expanded classifier: {old_classes} -> {new_classes}')


def build_incremental_subset(X_train, y_train, old_num_classes, replay_per_old_class, seed=42):
    rng = np.random.default_rng(seed)

    new_mask = y_train >= old_num_classes
    idx_new = np.where(new_mask)[0]

    idx_replay = []
    for c in range(old_num_classes):
        idx_c = np.where(y_train == c)[0]
        if len(idx_c) == 0:
            continue
        k = min(replay_per_old_class, len(idx_c))
        picked = rng.choice(idx_c, size=k, replace=False)
        idx_replay.extend(picked.tolist())

    idx_replay = np.array(idx_replay, dtype=np.int64)

    idx_final = np.concatenate([idx_new, idx_replay]) if len(idx_new) > 0 else idx_replay
    if len(idx_final) == 0:
        raise ValueError('Incremental subset is empty. Please check labels/data.')

    rng.shuffle(idx_final)
    summary = {
        'new_samples': int(len(idx_new)),
        'replay_samples': int(len(idx_replay)),
        'total_samples': int(len(idx_final)),
    }
    return X_train[idx_final], y_train[idx_final], summary


def make_class_weight(y):
    classes = np.unique(y)
    total = len(y)
    weights = {}
    for c in classes:
        count = int((y == c).sum())
        weights[int(c)] = total / (len(classes) * count) if count > 0 else 1.0
    return weights


def safe_split(X, y, train_size=0.75, random_state=42):
    try:
        return train_test_split(X, y, train_size=train_size, random_state=random_state, stratify=y)
    except Exception:
        print('Stratified split failed, fallback to random split.')
        return train_test_split(X, y, train_size=train_size, random_state=random_state)

# Load Dataset and Labels

In [4]:
labels = load_labels(LABEL_PATH)
NUM_CLASSES = len(labels)

X_dataset, y_dataset_raw, csv_files = load_sequence_dataset(CSV_GLOB, SEQUENCE_LENGTH, FEATURES_PER_FRAME)

# Convert 1-based labels to 0-based labels
y_dataset = y_dataset_raw.astype('int32') - 1

X_dataset, y_dataset = sanitize_dataset(X_dataset, y_dataset, NUM_CLASSES)

print(f'CSV files loaded: {len(csv_files)}')
print(f'Number of classes (label file): {NUM_CLASSES}')
print(f'Dataset shape: {X_dataset.shape}')
print(f'Unique labels in dataset: {np.unique(y_dataset)}')

Removed invalid-label samples: 1
CSV files loaded: 155
Number of classes (label file): 155
Dataset shape: (229335, 25, 80)
Unique labels in dataset: [  0   1   2   3   4   5   6   7   8   9  10  11  12  13  14  15  16  17
  18  19  20  21  22  23  24  25  26  27  28  29  30  31  32  33  34  35
  36  37  38  39  40  41  42  43  44  45  46  47  48  49  50  51  52  53
  54  55  56  57  58  59  60  61  62  63  64  65  66  67  68  69  70  71
  72  73  74  75  76  77  78  79  80  81  82  83  84  85  86  87  88  89
  90  91  92  93  94  95  96  97  98  99 100 101 102 103 104 105 106 107
 108 109 110 111 112 113 114 115 116 117 118 119 120 121 122 123 124 125
 126 127 128 129 130 131 132 133 134 135 136 137 138 139 140 141 142 143
 144 145 146 147 148 149 150 151 152 153 154]


# Decide Training Mode (Fresh / Resume / Expand)

In [5]:
existing_model = None
old_num_classes = 0

if os.path.exists(MODEL_SAVE_PATH):
    existing_model = tf.keras.models.load_model(MODEL_SAVE_PATH, compile=False)
    old_num_classes = int(existing_model.output_shape[-1])
    print(f'Existing model found: {MODEL_SAVE_PATH}, classes={old_num_classes}')
else:
    print('No existing model found. Will train from scratch.')

if existing_model is None:
    TRAIN_MODE = 'fresh'
elif old_num_classes == NUM_CLASSES:
    TRAIN_MODE = 'resume'
elif old_num_classes < NUM_CLASSES:
    TRAIN_MODE = 'expand'
else:
    raise ValueError(f'Existing model classes ({old_num_classes}) > current labels ({NUM_CLASSES}). Please verify label file/model compatibility.')

print('TRAIN_MODE:', TRAIN_MODE)

X_train_full, X_test, y_train_full, y_test = safe_split(X_dataset, y_dataset, train_size=0.75, random_state=RANDOM_SEED)

if TRAIN_MODE == 'expand':
    X_train, y_train, inc_summary = build_incremental_subset(
        X_train_full, y_train_full, old_num_classes, REPLAY_PER_OLD_CLASS, seed=RANDOM_SEED
    )
    print('Incremental subset summary:', inc_summary)
else:
    X_train, y_train = X_train_full, y_train_full

print(f'Train shape: {X_train.shape}, Test shape: {X_test.shape}')
print('Train finite:', np.isfinite(X_train).all(), 'Test finite:', np.isfinite(X_test).all())

Existing model found: model/keypoint_classifier/keypoint_sequence_classifier.keras, classes=155
TRAIN_MODE: resume
Train shape: (172001, 25, 80), Test shape: (57334, 25, 80)
Train finite: True Test finite: True


# Build / Load GRU Model

In [6]:
if NUM_CLASSES <= 0:
    raise ValueError('NUM_CLASSES must be > 0')

if TRAIN_MODE == 'fresh':
    model = build_gru_model(NUM_CLASSES, SEQUENCE_LENGTH, FEATURES_PER_FRAME)
elif TRAIN_MODE == 'resume':
    model = tf.keras.models.load_model(MODEL_SAVE_PATH, compile=False)
else:  # expand
    model = build_gru_model(NUM_CLASSES, SEQUENCE_LENGTH, FEATURES_PER_FRAME)
    copy_backbone_weights(existing_model, model)
    expand_classifier_weights(existing_model, model)

model.summary()

Model: "sequential_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 bidirectional_2 (Bidirecti  (None, 25, 256)           161280    
 onal)                                                           
                                                                 
 dropout_3 (Dropout)         (None, 25, 256)           0         
                                                                 
 bidirectional_3 (Bidirecti  (None, 128)               123648    
 onal)                                                           
                                                                 
 dropout_4 (Dropout)         (None, 128)               0         
                                                                 
 dense_2 (Dense)             (None, 64)                8256      
                                                                 
 batch_normalization_1 (Bat  (None, 64)               

# Compile and Train

In [7]:
cp_callback = tf.keras.callbacks.ModelCheckpoint(
    MODEL_SAVE_PATH,
    monitor='val_accuracy',
    mode='max',
    verbose=1,
    save_weights_only=False,
    save_best_only=True,
)

es_callback = tf.keras.callbacks.EarlyStopping(
    monitor='val_accuracy',
    mode='max',
    patience=20,
    restore_best_weights=True,
    verbose=1,
)

nan_callback = tf.keras.callbacks.TerminateOnNaN()
callbacks = [cp_callback, es_callback, nan_callback]

class_weight = make_class_weight(y_train)
print('Class weight keys:', sorted(class_weight.keys())[:10], '... total', len(class_weight))

history = None

if TRAIN_MODE == 'expand':
    for layer in model.layers:
        layer.trainable = (layer.name == 'classifier')

    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
                  loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])

    print('Phase 1: classifier warm-up')
    _ = model.fit(
        X_train, y_train,
        epochs=WARMUP_EPOCHS,
        batch_size=BATCH_SIZE,
        validation_data=(X_test, y_test),
        callbacks=callbacks,
        class_weight=class_weight,
        verbose=1,
    )

    for layer in model.layers:
        layer.trainable = True

    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=3e-4),
                  loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])

    print('Phase 2: full fine-tune')
    history = model.fit(
        X_train, y_train,
        epochs=FINETUNE_EPOCHS,
        batch_size=BATCH_SIZE,
        validation_data=(X_test, y_test),
        callbacks=callbacks,
        class_weight=class_weight,
        verbose=1,
    )
else:
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
                  loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])

    history = model.fit(
        X_train, y_train,
        epochs=FINETUNE_EPOCHS,
        batch_size=BATCH_SIZE,
        validation_data=(X_test, y_test),
        callbacks=callbacks,
        class_weight=class_weight,
        verbose=1,
    )

Class weight keys: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9] ... total 155
Epoch 1/120
1303/5376 [======>.......................] - ETA: 3:39 - loss: 0.0099 - accuracy: 0.9971

KeyboardInterrupt: 

# Save and Export (Keras / SavedModel / TFLite)

In [ ]:
os.makedirs(Path(MODEL_SAVE_PATH).parent, exist_ok=True)
model.save(MODEL_SAVE_PATH)
print(f'Keras model saved: {MODEL_SAVE_PATH}')

os.makedirs(SAVED_MODEL_DIR, exist_ok=True)
model.save(SAVED_MODEL_DIR)
print(f'SavedModel exported: {SAVED_MODEL_DIR}')

converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.target_spec.supported_ops = [
    tf.lite.OpsSet.TFLITE_BUILTINS,
    tf.lite.OpsSet.SELECT_TF_OPS,
]
converter._experimental_lower_tensor_list_ops = False
tflite_model = converter.convert()

with open(TFLITE_SAVE_PATH, 'wb') as f:
    f.write(tflite_model)

print(f'TFLite model saved: {TFLITE_SAVE_PATH}')

# Evaluate

In [ ]:
val_loss, val_acc = model.evaluate(X_test, y_test, verbose=0)
print(f'Validation Loss: {val_loss:.6f}, Validation Accuracy: {val_acc:.6f}')

# Quick TFLite Inference Test

In [ ]:
interpreter = tf.lite.Interpreter(model_path=TFLITE_SAVE_PATH)
interpreter.allocate_tensors()

input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

sample = np.array([X_test[0]], dtype=np.float32)
interpreter.set_tensor(input_details[0]['index'], sample)
interpreter.invoke()
result = interpreter.get_tensor(output_details[0]['index'])

pred_idx = int(np.argmax(result[0]))
true_idx = int(y_test[0])
print('Predicted index (0-based):', pred_idx, 'Label:', labels[pred_idx] if pred_idx < len(labels) else 'N/A')
print('Actual index (0-based):', true_idx, 'Label:', labels[true_idx] if true_idx < len(labels) else 'N/A')

# Plot Training Curves

In [ ]:
import matplotlib.pyplot as plt

if history is None:
    print('No training history found.')
else:
    hist = history.history
    epochs = range(1, len(hist.get('accuracy', [])) + 1)

    plt.figure(figsize=(12, 5))

    plt.subplot(1, 2, 1)
    plt.plot(epochs, hist.get('accuracy', []), label='train_accuracy')
    plt.plot(epochs, hist.get('val_accuracy', []), label='val_accuracy')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy')
    plt.title('Training vs Validation Accuracy')
    plt.legend()
    plt.grid(alpha=0.3)

    plt.subplot(1, 2, 2)
    plt.plot(epochs, hist.get('loss', []), label='train_loss')
    plt.plot(epochs, hist.get('val_loss', []), label='val_loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.title('Training vs Validation Loss')
    plt.legend()
    plt.grid(alpha=0.3)

    plt.tight_layout()
    plt.show()